In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [ ]:
# Task 3.1: Features, Train-Test Split and Baseline

# Load cleaned dataset
df = pd.read_csv("appointments_clean.csv")

# Keep appointment IDs separately for later risk-score export
appointment_ids = df["appointment_id"]

# Select required features
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# Convert reminder_sent Yes/No into 1/0
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

# One-hot encode booking channel and visit type
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

# Target variable
y = df["no_show"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

baseline_predictions = np.zeros(len(y_test))

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Feature shape:", X.shape)
print("Training size:", len(X_train))
print("Test size:", len(X_test))
print("No-shows in test set:", int(y_test.sum()))
print("Baseline accuracy:", round(baseline_accuracy, 4))

Feature shape: (990, 11)
Training size: 792
Test size: 198
No-shows in test set: 49
Baseline accuracy: 0.7525


In [ ]:
# Task 3.2: Logistic Regression

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Scale the features
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

# Predictions
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

# Evaluation metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

# Feature coefficients
print("\nFeature Coefficients:")

for feature, coef in zip(X.columns, model.coef_[0]):
    print(feature, round(coef, 4))

# Reminder odds ratio (per one standard deviation)
reminder_idx = list(X.columns).index("reminder_sent")

reminder_coef = model.coef_[0][reminder_idx]
reminder_sd = scaler.scale_[reminder_idx]

print("\nReminder OR (per 1 SD):", round(np.exp(reminder_coef), 4))
print("Reminder OR (No to Yes):", round(np.exp(reminder_coef / reminder_sd), 4))

Accuracy: 0.7879
Confusion Matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7889

Feature Coefficients:
age -0.2539
distance_km 0.2534
fee_inr 0.4528
lead_days 0.6145
prior_no_shows 0.6496
reminder_sent -0.5807
booking_channel_App -0.0468
booking_channel_Phone 0.2159
booking_channel_Walk-In -0.1864
visit_type_Consultation 0.0157
visit_type_Follow-up -0.0157

Reminder OR (per 1 SD): 0.5595
Reminder OR (No to Yes): 0.3094


In [ ]:
# Task 3.3: Decision Tree and Overfitting

from sklearn.tree import DecisionTreeClassifier

# Full-depth decision tree
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

# Depth-3 decision tree
dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_3.fit(X_train, y_train)


print("Full-depth Tree")
print("Training Accuracy:", round(dt_full.score(X_train, y_train), 4))
print("Test Accuracy:", round(dt_full.score(X_test, y_test), 4))

# Depth-3 tree accuracy
print("\nDepth-3 Tree")
print("Training Accuracy:", round(dt_3.score(X_train, y_train), 4))
print("Test Accuracy:", round(dt_3.score(X_test, y_test), 4))

Full-depth Tree
Training Accuracy: 1.0
Test Accuracy: 0.6717

Depth-3 Tree
Training Accuracy: 0.822
Test Accuracy: 0.7778


#The full-depth decision tree achieved perfect training accuracy but much lower test accuracy, even below the baseline, indicating overfitting. Restricting the tree to depth 3 reduced overfitting and improved generalisation on unseen data.

In [ ]:
# Task 3.4: Random Forest and Model Comparison

from sklearn.ensemble import RandomForestClassifier

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Predictions
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

# Random Forest evaluation
print("Random Forest")
print("Training Accuracy:", round(rf.score(X_train, y_train), 4))
print("Test Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("Recall:", round(recall_score(y_test, rf_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, rf_prob), 4))

# Top 3 feature importances
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

print("\nTop 3 Feature Importances:")
print(importance_df.head(3))


comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Depth-3 Tree", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        dt_3.score(X_test, y_test),
        accuracy_score(y_test, rf_pred)
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, dt_3.predict(X_test)),
        recall_score(y_test, rf_pred)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob),
        roc_auc_score(y_test, dt_3.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, rf_prob)
    ]
})

print("\nModel Comparison:")
print(comparison.round(4))

Random Forest
Training Accuracy: 1.0
Test Accuracy: 0.7475
Recall: 0.3061
ROC-AUC: 0.7057

Top 3 Feature Importances:
       feature  importance
3    lead_days    0.219905
0          age    0.196788
1  distance_km    0.194739

Model Comparison:
                 Model  Accuracy  Recall  ROC-AUC
0  Logistic Regression    0.7879  0.3469   0.7889
1         Depth-3 Tree    0.7778  0.2245   0.6997
2        Random Forest    0.7475  0.3061   0.7057


#Task 3.5 — Model Selection, Threshold Tuning and Final Evaluation
Use training-set cross-validation to select the model and threshold. Evaluate the selected model once on the test set and export test risk scores.


In [ ]:
# Task 3.5 (a & b)

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, cross_val_score

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Grid Search for Random Forest
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)

grid.fit(X_train, y_train)

print("Best RF Parameters:", grid.best_params_)
print("Tuned RF CV ROC-AUC:", round(grid.best_score_, 4))

# Cross-validated ROC-AUC on training data
lr_cv = cross_val_score(
    lr_pipe, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

dt3_cv = cross_val_score(
    dt_3, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

rf_cv = cross_val_score(
    rf, X_train, y_train, cv=5, scoring="roc_auc"
).mean()

tuned_rf_cv = grid.best_score_

print("\nTraining CV ROC-AUC:")
print("Logistic Regression:", round(lr_cv, 4))
print("Depth-3 Tree:", round(dt3_cv, 4))
print("Random Forest:", round(rf_cv, 4))
print("Tuned Random Forest:", round(tuned_rf_cv, 4))

# Select the winner using training CV only
models_cv = {
    "Logistic Regression": lr_cv,
    "Depth-3 Tree": dt3_cv,
    "Random Forest": rf_cv,
    "Tuned Random Forest": tuned_rf_cv
}

winner_name = max(models_cv, key=models_cv.get)

if winner_name == "Logistic Regression":
    winner_model = lr_pipe
elif winner_name == "Depth-3 Tree":
    winner_model = dt_3
elif winner_name == "Random Forest":
    winner_model = rf
else:
    winner_model = grid.best_estimator_

print("\nSelected Model:", winner_name)

Best RF Parameters: {'max_depth': 3, 'n_estimators': 200}
Tuned RF CV ROC-AUC: 0.7673

Training CV ROC-AUC:
Logistic Regression: 0.7737
Depth-3 Tree: 0.6784
Random Forest: 0.7466
Tuned Random Forest: 0.7673

Selected Model: Logistic Regression


In [ ]:
# Task 3.5 (c): Threshold selection using training OOF probabilities

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

# Out-of-fold probabilities from training data only
oof = cross_val_predict(
    winner_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

# Compare thresholds
threshold_results = []

for threshold in [0.5, 0.35]:
    oof_pred = (oof >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_train, oof_pred),
        "recall": recall_score(y_train, oof_pred),
        "f1": f1_score(y_train, oof_pred),
        "flagged": int(oof_pred.sum())
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4))

# Choose threshold with higher F1
chosen_threshold = threshold_df.loc[
    threshold_df["f1"].idxmax(), "threshold"
]

print("\nChosen Threshold:", chosen_threshold)

   threshold  precision  recall      f1  flagged
0       0.50     0.5753  0.2485  0.3471       73
1       0.35     0.5130  0.4675  0.4892      154

Chosen Threshold: 0.35


In [ ]:
# Task 3.5 (d): Final evaluation on test data

winner_model.fit(X_train, y_train)

# Test probabilities
test_probs = winner_model.predict_proba(X_test)[:, 1]

test_flags = (test_probs >= chosen_threshold).astype(int)

print("Selected Model:", winner_name)
print("Chosen Threshold:", chosen_threshold)

print("Test Precision:", round(precision_score(y_test, test_flags), 4))
print("Test Recall:", round(recall_score(y_test, test_flags), 4))
print("Test F1:", round(f1_score(y_test, test_flags), 4))
print("Flagged Appointments:", int(test_flags.sum()))
print("Test ROC-AUC:", round(roc_auc_score(y_test, test_probs), 4))

Selected Model: Logistic Regression
Chosen Threshold: 0.35
Test Precision: 0.5714
Test Recall: 0.5714
Test F1: 0.5714
Flagged Appointments: 49
Test ROC-AUC: 0.7889


In [ ]:
# Task 3.5 (e): Save test risk scores

test_risk_scores = pd.DataFrame({
    "appointment_id": appointment_ids.loc[X_test.index].values,
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_flags
})

test_risk_scores.to_csv("test_risk_scores.csv", index=False)

print("File saved: test_risk_scores.csv")
print("Rows:", len(test_risk_scores))
print("Flagged rows:", int(test_risk_scores["flag"].sum()))

File saved: test_risk_scores.csv
Rows: 198
Flagged rows: 49
